# 10 · AutoGen — conversational multi-agent
Writer ↔ Critic iterate until *APPROVE*.

*Note: notebooks have a running event loop, so we `await` directly (no `asyncio.run`).* 

In [ ]:
# Bootstrap: make the repo root importable so `import config` works from notebooks/
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from config import assert_key
assert_key()
print("Gateway ready.")

In [ ]:
from config import FAST_MODEL, API_KEY, BASE_URL
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model=FAST_MODEL, base_url=BASE_URL, api_key=API_KEY,
    model_info={"vision": False, "function_calling": True, "json_output": True,
                "family": "unknown", "structured_output": False})

writer = AssistantAgent("writer", model_client=model_client,
    system_message="You write punchy product taglines. Revise on feedback.")
critic = AssistantAgent("critic", model_client=model_client,
    system_message="Critique the tagline in one line. Reply with the single word APPROVE once excellent.")
team = RoundRobinGroupChat([writer, critic],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(8))

In [ ]:
await Console(team.run_stream(task="A tagline for the Acme Sentinel security robot."))
await model_client.close()

## 🧪 Your turn
1. Add a third **editor** agent that trims the winning tagline to <=6 words.
2. Change the termination to `MaxMessageTermination(4)` — how does output quality change?

In [ ]:
# your code here